In [21]:
import numpy as np
import pyvista as pv


def plot_data(
        data: np.ndarray | list,
        size: float = 5.0,
):
    coords = data
    colors = np.clip(data, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_clustered_pv(
        data: np.ndarray,
        labels: np.ndarray,
        cluster_centers: np.ndarray,
        size: float = 4.0
):
    coords = data
    cluster_colors = cluster_centers[labels]
    colors = np.clip(cluster_colors, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_centroids_pv(
        cluster_centers: np.ndarray,
        size: float = 18.0
):
    coords = cluster_centers
    colors = np.clip(cluster_centers, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size,
        render_points_as_spheres=True
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()

In [22]:
import numpy as np

def generate_dataset(centers, stds, n_samples=500, ndim=3):
    X = []
    y = []

    for i, center in enumerate(centers):
        cluster = np.random.normal(loc=center, scale=stds[i], size=(n_samples // len(centers), ndim))
        X.append(cluster)
        y.append(np.full(n_samples // len(centers), i))

    return np.vstack(X), np.concatenate(y)

In [23]:
import pandas as pd
import pyvista as pv
import matplotlib.pyplot as plt

default_colors = np.array(plt.colormaps.get_cmap('tab10').colors)

In [24]:
data, ground_truth = generate_dataset(
    [[1, 1, 1], [3, 3, 3], [2, 2, 1]],
    stds=[.5, .5, .5],
    n_samples=500
)

pd.DataFrame(data)

,0,1,2
0,2.309461,1.203306,1.110655
1,1.949402,0.331025,0.917487
2,1.112158,0.288663,0.212439
3,1.384812,0.805727,1.093657
4,1.074792,1.956611,0.661288
...,...,...,...
493,1.354514,2.016756,1.127929
494,1.654748,2.792423,1.033548
495,2.667255,1.795222,0.442704
496,3.132661,2.661647,0.767587


In [25]:
plt = pv.Plotter()

for n, d in zip(ground_truth, data):
    cloud = pv.PolyData(d)
    color = default_colors[n % len(default_colors)]

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=10
    )

plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42957/index.html?ui=P_0x7f9819b1a210_7&reconnect=auto" class="pyvi…

In [26]:
from ktree.ntree import NTreeDynamic

tree = NTreeDynamic(1)

for a in data:
    tree.insert(a)

sorted_data = tree.sort()

In [27]:
plt = pv.Plotter()

for n, cluster in enumerate(sorted_data):
    s_data = list(cluster)
    cloud = pv.PolyData(s_data)
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=5
    )

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=len(s_data) // 2
    )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42957/index.html?ui=P_0x7f9819b1a710_8&reconnect=auto" class="pyvi…

In [28]:
n_clusters = 2

all_clusters = sorted(sorted_data, key=lambda x: len(x))[::-1]
clusters = all_clusters[:n_clusters]
all_data = all_clusters[n_clusters:]

main_clusters = []

for cluster in clusters:
    c_centroid = np.mean([*cluster], axis=0)
    sum_dist = 0

    sub_cluster = []

    for data in all_data:
        d_centroid = np.mean([*data], axis=0)
        dist = np.linalg.norm(c_centroid - d_centroid)

        sum_dist += dist

        sub_cluster.append((dist, data))

    mead_dist = sum_dist / len(all_data)
    sub_cluster = sorted(sub_cluster, key=lambda a: a[0])

    main_clusters.append((cluster, [data for (dist, data) in sub_cluster if dist < mead_dist]))


for (cluster, all_data) in main_clusters:
    print("Cluster: ", cluster)
    print("Data: ", all_data)


Cluster:  Cluster(axis=[[-0.7017870758579057, 1.8761524851002913], [-0.5354872824594594, 1.956611392885172], [-0.11390809546217184, 1.9719063763881914]])
Data:  [Cluster(axis=[[0.45742003348438154, 1.8639856902551653], [1.996255720389725, 3.0484705931084326], [-0.15322670714403874, 1.938913354001933]]), Cluster(axis=[[1.9001533624969693, 3.019261648948727], [0.1643641815677479, 1.9607700912732045], [-0.14939237139538575, 1.8620983994484648]]), Cluster(axis=[[1.3902843249338626, 1.7382592014868847], [1.1166970507092269, 1.5697386254884638], [2.306287080207713, 2.323972805208295]])]
Cluster:  Cluster(axis=[[2.0525096329652146, 4.482203603623768], [1.9945081921063608, 4.48432862352303], [2.0382665720781836, 4.306010173283402]])
Data:  [Cluster(axis=[[2.0006490300781907, 3.831945469186887], [1.2316415156258032, 1.9538297001875606], [2.0257356258013193, 3.58133071032518]]), Cluster(axis=[[1.7926395622351, 1.7926395622351], [2.9411946133036677, 2.9411946133036677], [2.398724449625079, 2.3987

In [29]:
plt = pv.Plotter()

for n, (cluster, all_data) in enumerate(main_clusters):
    s_data = list(cluster)

    cloud = pv.PolyData(list(cluster))
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=25
    )

    for data in (all_data + [cluster]):
        cloud = pv.PolyData(list(data))

        plt.add_points(
            cloud,
            color=color,
            render_points_as_spheres=True,
            smooth_shading=True,
            point_size=10
        )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42957/index.html?ui=P_0x7f9819b19a90_9&reconnect=auto" class="pyvi…